# `H+E` Evaluation Funnel Walkthrough

This notebook demonstrates how to call each stage of the `H+E` hybrid judge in Python and inspect the exact inputs and outputs at every step:
1. **Inspecting the Inputs**: Question, Ground Truth SQL & Result, Candidate SQL(s) (including multi-step traces), and Ground Truth Label.
2. **Stage 1 (`align_tables`, `normalize_sql_n`)**: Deterministic SQL normalization, column bijection, and row matching.
3. **Stages 0–4 (`judge_item(..., verbose=True)`)**: Full `H+E` trace showing where deterministic rules short-circuit vs. where items progress to Jev/Gemini, including examples where the judge is **correct** and where it **abstains / differs**.

In [1]:
import json
from pathlib import Path

from txt2sql_eval_harness.alignment import align_tables
from txt2sql_eval_harness.baselines import judge_b2
from txt2sql_eval_harness.clients import CallContext
from txt2sql_eval_harness.codec import decode_labelled_item
from txt2sql_eval_harness.judge import judge_item
from txt2sql_eval_harness.models import Contract, EvalConfig
from txt2sql_eval_harness.sql_analysis import normalize_sql_n

TARGETS = {
    "S2__bird_minidev:1471__agg_swap__0",      # sql_only scalar bug (H+E correct via rule:D4)
    "S2__bird_minidev:1457__agg_swap__0",      # sql_only multi-column table (H+E correct via rule:D2)
    "S3m__bird_minidev:1471__early_support",  # full_trace multi-step (H+E abstains offline; B2 wrong)
}

items = {}
with Path("../data/items.jsonl").open() as f:
    for line in f:
        item = decode_labelled_item(json.loads(line))
        if item.item_id in TARGETS:
            items[item.item_id] = item

for item_id, item in items.items():
    print(f"================ {item_id} (GT label: {item.label}) ================")
    print("Question:", item.trace.question)
    print("GT SQL:  ", item.gt.sql_chain[-1])
    for st in item.trace.steps:
        print(f"Step {st.index} SQL:", st.sql)
        print(f"Step {st.index} rows:", len(st.result.rows) if st.result else None)
    if item.trace.final_answer:
        print("Final Answer:", item.trace.final_answer[:200])
    print()

================ S2__bird_minidev:1471__agg_swap__0 (GT label: incorrect) ================
Question: What is the ratio of customers who pay in EUR against customers who pay in CZK?
GT SQL:   SELECT CAST(SUM(IIF(Currency = 'EUR', 1, 0)) AS FLOAT) / SUM(IIF(Currency = 'CZK', 1, 0)) AS ratio FROM customers
Step 1 SQL: SELECT CAST(SUM(IIF(Currency = 'EUR', 1, 0)) AS REAL) / COUNT(IIF(Currency = 'CZK', 1, 0)) AS ratio FROM customers
Step 1 rows: 1

================ S2__bird_minidev:1457__agg_swap__0 (GT label: incorrect) ================
Question: Give the full name and contact number of members who had to spend more than average on each expense.
GT SQL:   SELECT DISTINCT T3.first_name, T3.last_name, T3.phone FROM expense AS T1 INNER JOIN budget AS T2 ON T1.link_to_budget = T2.budget_id INNER JOIN member AS T3 ON T3.member_id = T1.link_to_member WHERE T1.cost > ( SELECT AVG(T1.cost) FROM expense AS T1 INNER JOIN budget AS T2 ON T1.link_to_budget = T2.budget_id INNER JOIN member AS T3 ON T3.

## Stage 1: Deterministic SQL Normalization & Table Alignment
Before any LLM is called, `normalize_sql_n` checks syntactic equivalence (`D1`), and `align_tables` computes a 1-to-1 column bijection and row matching between the ground-truth `ResultTable` and the candidate `ResultTable`.

In [3]:
for item_id, item in items.items():
    gt_sql = item.gt.sql_chain[-1]
    cand_sql = item.trace.steps[-1].sql
    gt_res = item.gt.results[-1]
    cand_res = item.trace.steps[-1].result

    aln, row_diff, flags = align_tables(gt_res, cand_res, Contract(order_required=False, limit_required=False, gt_columns=gt_res.columns))
    print(f"=== {item_id} ===")
    print("  GT SQL norm == Cand SQL norm:", normalize_sql_n(gt_sql) == normalize_sql_n(cand_sql))
    print("  Alignment relation:", aln.relation, "| col_map:", aln.column_map)
    print("  Flags:", flags)

=== S2__bird_minidev:1471__agg_swap__0 ===
  GT SQL norm == Cand SQL norm: False
  Alignment relation: disjoint | col_map: {0: None}
  Flags: []
=== S2__bird_minidev:1457__agg_swap__0 ===
  GT SQL norm == Cand SQL norm: False
  Alignment relation: gen_subset_gt | col_map: {0: None, 1: None, 2: None}
  Flags: []
=== S3m__bird_minidev:1471__early_support ===
  GT SQL norm == Cand SQL norm: False
  Alignment relation: disjoint | col_map: {0: None}
  Flags: []


## Stages 0–4: Full `judge_item(..., method="H+E", verbose=True)`
Using `verbose=True` prints each stage the harness goes through:
- On the two `sql_only` `S2` items, `H+E` **correctly** identifies `INCORRECT` in Stage 1 (`rule:D4` and `rule:D2`) at `$0.00` cost.
- On the multi-step `full_trace` `S3m` item (GT label `correct`), `H+E` progresses through Stage 1 and Stage 2 (`EvidencePacket`) to Stage 3 (`ABSTAIN` offline because `allow_remote=False`), whereas `B2` **incorrectly** grades it `INCORRECT` because `B2` only checks `steps[-1]`.

In [5]:
cfg = EvalConfig(local_only=True)
ctx = CallContext(cfg=cfg, allow_remote=True)

for item_id, item in items.items():
    print(f"--- {item_id} (GT: {item.label.upper()}) ---")
    verdict = judge_item(item.trace, item.gt, method="H+E", ctx=ctx, verbose=True)
    b2_verdict = judge_b2(item.trace, item.gt, cfg=cfg)
    print(f"  H+E Verdict: {verdict.label} (decided_by={verdict.decided_by}, reason={verdict.reason})")
    print(f"  B2  Verdict: {b2_verdict.label} (decided_by={b2_verdict.decided_by}, reason={b2_verdict.reason})\n")

--- S2__bird_minidev:1471__agg_swap__0 (GT: INCORRECT) ---
[Method: H+E] Full hybrid funnel: Stage 0-1 deterministic rules + Stage 2 packet + Stage 3 Jev (Q1-Q6) + Stage 4 Gemini escalation
[Stage 0-1] mode=sql_only, alignment=disjoint, claims=0, derivations=0, flags=[], rule_hit=rule:D4
[Stage 1] Short-circuit rule:D4 -> INCORRECT (scalar_counterexample)
  H+E Verdict: INCORRECT (decided_by=rule:D4, reason=scalar_counterexample)
  B2  Verdict: INCORRECT (decided_by=baseline, reason=None)

--- S2__bird_minidev:1457__agg_swap__0 (GT: INCORRECT) ---
[Method: H+E] Full hybrid funnel: Stage 0-1 deterministic rules + Stage 2 packet + Stage 3 Jev (Q1-Q6) + Stage 4 Gemini escalation
[Stage 0-1] mode=sql_only, alignment=gen_subset_gt, claims=0, derivations=0, flags=[], rule_hit=rule:D2
[Stage 1] Short-circuit rule:D2 -> INCORRECT (no_data)
  H+E Verdict: INCORRECT (decided_by=rule:D2, reason=no_data)
  B2  Verdict: INCORRECT (decided_by=baseline, reason=None)

--- S3m__bird_minidev:1471__early